In [ ]:
# --- path configuration -------------------------------------------------
# Every path goes through resolve() (inputs under data/) or out_path() (this run's output).
# Set SURFACEOME_DATA to point at the data archive; see README.md.
from surfaceome_config import resolve, out_path


In [ ]:
import pandas as pd
import numpy as np

from surfaceomeTopography import *

from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
#from Bio.Alphabet import IUPAC

%load_ext autoreload

%autoreload 2

In [ ]:
est_homology = pd.read_csv(resolve('database/height_estimates_homology.csv'))
est_af = pd.read_csv(resolve('database/height_estimates_alphafold.csv'))
est_dom = pd.read_csv(resolve('database/height_estimates_domain_counting.csv'))

### Load pdb to uniprot mappings downloaded from RSCB PDB (human)
downloaded using the advanced search tool (**search related features** https://www.rcsb.org/pages/search_features#search_advanced-search) limiting to human and uniprot id within human surfaceome list. The sequence report was then downloaded


*NOTE: used alternative method of SIFTS database*

## SIFTS mappings

https://www.ebi.ac.uk/pdbe/docs/sifts/index.html
"SIFTS: updated Structure Integration with Function, Taxonomy and Sequences resource allows 40-fold increase in coverage of structure-based annotations for proteins"

In [ ]:
sifts = pd.read_csv(str(resolve('inputs/structures/pdb_chain_uniprot.tsv')), skiprows = [0],
            sep = '\t')


sifts_surf = pd.merge(sifts, est_homology[['ID','ID link_first', 'seq_start', 'seq_end']], left_on = 'SP_PRIMARY', right_on = 'ID link_first')


sifts_surf = sifts_surf.assign(ID_link = sifts_surf['ID'].str.split('[').str[0])
sifts_surf = sifts_surf.assign(start = sifts_surf['ID'].str.split('[').str[1])
sifts_surf = sifts_surf.assign(start = sifts_surf['start'].str.split(':').str[0])
sifts_surf = sifts_surf.assign(end = sifts_surf['ID'].str.split(':').str[1].str.strip(']'))
sifts_surf.rename(columns={'ID_link': 'ID link'}, inplace=True)
sifts_surf.start = pd.to_numeric(sifts_surf.start)
sifts_surf.end = pd.to_numeric(sifts_surf.end)
sifts_surf.PDB_BEG = sifts_surf.PDB_BEG.replace('None', np.nan)
sifts_surf.PDB_END = sifts_surf.PDB_END.replace('None', np.nan)

sifts_surf.head()

In [ ]:
#classify those that contain the extracellular domain
sifts_surf = sifts_surf.assign(extracellular = np.where(sifts_surf.SP_BEG >= sifts_surf.start-4, #buffer, 
                                           np.where(sifts_surf.SP_END <= sifts_surf.end+4, True, False), False))

sifts_surf = sifts_surf[sifts_surf.extracellular]

sifts_surf = sifts_surf.assign(pdb_len = sifts_surf.SP_END-sifts_surf.SP_BEG)
sifts_surf = sifts_surf.assign(extracellular_len = sifts_surf.end-sifts_surf.start)

In [ ]:
sifts_surf = sifts_surf.assign(ecd_start = sifts_surf[['SP_BEG', 'start']].max(axis=1))
sifts_surf = sifts_surf.assign(ecd_end = sifts_surf[['SP_END', 'end']].min(axis=1))

#difference between PBD start and end and ECD start and END
sifts_surf = sifts_surf.assign(start_diff = sifts_surf['ecd_start']-sifts_surf['SP_BEG'])
sifts_surf = sifts_surf.assign(end_diff = sifts_surf['SP_END']-sifts_surf['ecd_end'])

#fill na for pdb mapping
#sifts_surf.PDB_BEG.replace('1A', '1', inplace = True)
#sifts_surf.PDB_BEG.replace('43P', '1', inplace = True)
#sifts_surf.PDB_BEG.replace('45P', '1', inplace = True)
#sifts_surf.PDB_BEG.replace('46P', '1', inplace = True)
#sifts_surf.PDB_BEG.replace('0B', '0', inplace = True)

sifts_surf['PDB_BEG'] = pd.to_numeric(sifts_surf['PDB_BEG'].fillna(sifts_surf['RES_BEG']), errors='coerce')
sifts_surf['PDB_END'] = pd.to_numeric(sifts_surf['PDB_END'].fillna(sifts_surf['RES_END']), errors='coerce')

#mapped PDB residues
sifts_surf = sifts_surf.assign(MAP_BEG = sifts_surf['PDB_BEG']+sifts_surf['start_diff'])
sifts_surf = sifts_surf.assign(MAP_END = sifts_surf['PDB_END']-sifts_surf['end_diff'])

#len of ECD in struct
sifts_surf = sifts_surf.assign(ecd_len = sifts_surf['ecd_end']-sifts_surf['ecd_start'])

In [ ]:
sifts_surf = sifts_surf.assign(percent_struct =  sifts_surf.ecd_len/ sifts_surf.extracellular_len)
sifts_surf

In [ ]:
#export pdbs to measure in pymol
pdbs_for_measurement = sifts_surf[sifts_surf.percent_struct > 0.7][['ID link','PDB', 'CHAIN','MAP_BEG', 'MAP_END','PDB_BEG', 'PDB_END', 'SP_BEG', 'SP_END', 'start', 'end']]
pdbs_for_measurement.to_csv(out_path('database/surfaceome_pdbs.csv'), index = False)

# Import structure measurements

In [ ]:
#import structural measurements taken in pymol

#perform measurements in pymol
#import results
measured = pd.read_csv(str(resolve('measurements/pdb_structure_sizes.txt')), sep = '\t', header= None,
            names = ['CHAIN', 'PDB', 'PDB_len', 'x', 'y', 'z'])

#cleanup any duplicate measurements
measured = measured.drop_duplicates(subset = ['CHAIN', 'PDB'])

#merge measurements and PBD info
measured_struct = pd.merge(pdbs_for_measurement.drop_duplicates(subset = ['CHAIN', 'PDB']), measured, on = ['PDB', 'CHAIN'])

#get ecd len
measured_struct['seq_len'] = measured_struct['end']-measured_struct['start']

#filter structures that are longer than the ecd len
#only want to do this because I didn't select the residues when measureing
#SHOULD select residues...
#measured_struct = measured_struct[measured_struct['PDB_len'] <= measured_struct['seq_len']]

#calc percent of ECD measured...for full structures!!
measured_struct['frac_modeled'] = measured_struct['PDB_len']/measured_struct['seq_len']

#max dim/min dimensions of the structure
measured_struct = find_dims(measured_struct)

In [ ]:
#summarize strutures, groupby protein id then average measurements?
#there are multiple structural measurements per prot often...

#Require the chosen structure to actually RESOLVE 70% of the ectodomain, not merely to be
#mapped across it.  percent_struct (cell 9) is computed from the SIFTS SP range, and a cryo-EM
#entry routinely maps a full-length chain while resolving a fraction of it -- 8ufb/X is mapped
#over all 770 residues of the VLDLR ectodomain and resolves 78.  Since this table overrides
#the model-based height wherever a protein appears in it, an unresolved structure replaces a
#26 nm estimate with 5.9 nm.  frac_modeled is PDB_len/seq_len, already computed at cell 21,
#and 0.7 is the same threshold used on the mapped range.
resolved = measured_struct[measured_struct.frac_modeled >= 0.7]

unique_pdbs = resolved.sort_values('PDB_len', ascending = False).groupby('ID link')\
       .agg({'PDB': 'first', 'CHAIN':'first', 'PDB_len':'first', 'max_dim': 'first', 'min_dim': 'first',
            'start': 'first', 'end':'first', 'SP_BEG':'first', 'SP_END': 'first'})\
       .reset_index().dropna().sort_values('max_dim')


In [ ]:
unique_pdbs.to_csv(out_path('database/pdb_structures_measured.csv'), index = False)

In [ ]:
#proteins modeled well with homology...
homology_model = est_homology[est_homology['methods_<lambda>'] == 'homology'].drop(columns = ['index']).copy()

#check to see is homology model matches a measured structure...
#extract PBD ID
homology_model.loc[:,'PDB']= homology_model['Hit_first'].str.split(pat= '_', expand = True)[0].str.slice(start = 1, stop = 5)#.str.upper()

#extract chain ID
homology_model.loc[:,'CHAIN']= homology_model['Hit_first'].str.split(pat= '_', expand = True)[0].str.slice(start = 5, stop = 6).str.upper()

homology_model.dropna(axis= 1, inplace = True)
homology_model.rename(columns = {'ID link_first': 'ID link', 'seq_len_first': 'seq_len', 
                                'methods_<lambda>': 'methods' ,'max_dim_first': 'max_dim',
                                 'min_dim_first': 'min_dim', 'Job id_first': 'Job id',
                                 'Hit_first': 'Hit', 'Confidence (%)_first': 'Confidence (%)',
                                 'Sequence identity (%)_first': 'Sequence identity (%)',
                                 'source_last': 'source'
                                }, inplace = True)

homology_model.dropna(axis= 1, inplace = True, how = 'all')


In [ ]:
#merge with measured unique structures
homology_structures = homology_model.merge(unique_pdbs, on = ['ID link', 'PDB', 'CHAIN'], suffixes = ['_homology', '_struct'])

#homology models that are already measured as structures
model_structures = homology_model.merge(measured_struct, on = ['ID link', 'PDB', 'CHAIN'], suffixes = ['_homology', '_struct'])

In [ ]:
s_and_h = homology_model.merge(unique_pdbs, on = ['ID link'], how = 'outer', indicator = True, suffixes = ['_homology', '_struct'])

In [ ]:
comparisons = s_and_h[s_and_h._merge =='both']

In [ ]:
not_struct = comparisons[~comparisons.ID.isin(model_structures.ID)]

In [ ]:
#set scale to mimic seq len plot?
import plotly.express as px

df = not_struct.sort_values('PDB_len', ascending= False).drop_duplicates('ID link').copy()

df["max_dim_struct"] = df["max_dim_struct"]/10.0

df["max_dim_homology"] = df["max_dim_homology"]/10.0


fig = px.scatter(df, x="max_dim_struct", y="max_dim_homology", #color="source",
                 size='seq_len', 
                 color = 'Sequence identity (%)', color_continuous_scale = 'plasma_r', hover_data=['PDB_homology', 'ID', 'PDB_struct'], 
                 trendline="ols",
                 
                     labels={
                     "max_dim_struct": "structural height (nm)",
                     "max_dim_homology": "homology model height (nm)",
                     "mean_confidence_trimmed_first": "mean model confidence"
                 },
                title = 'Homology Models vs. Structural Measurement',
                 
                 
                 width=700, height=600
                 
                 
                )
fig.update_layout(yaxis=dict(range=[0,30]), xaxis=dict(range=[0,30]))

fig.show()

results = px.get_trendline_results(fig)
print(results)

fig.write_image(out_path('figures/S2_methods_vs_structures.pdf'))
results.px_fit_results.iloc[0].summary()

#fig.write_image(out_path('figures/S2_methods_vs_structures.pdf'))

# structures vs alphafold 
**what structures were previously used in the modeling??**


## What information does AlphaFold use from the Protein Data Bank?
 
AlphaFold is trained on protein chains in the PDB released before 2018-04-30. Predictions can also make use of up to 4 templates released before 2021-02-15. However, templates are not a critical input for AlphaFold to make an accurate prediction; the model can make a strong prediction based on a multiple sequence alignment alone.
 


# not nessary

In [ ]:
#the heights are predictions, so this selects the ectodomains whose height is model-derived
#with negligible disorder -- the cleanest comparison against a solved measurement
af_only = est_af[(est_af['methods_<lambda>']== 'alphafold')|(est_af['methods_<lambda>']== 'alphafold,disorder')].rename(columns = {'af_ID_first': 'ID link', 'seq_len_first': 'seq_len', 
                                'methods_<lambda>': 'methods' ,'max_dim_first': 'max_dim',
                                 'min_dim_first': 'min_dim', 'Job id_first': 'Job id',
                                 'Hit_first': 'Hit', 'Confidence (%)_first': 'Confidence (%)',
                                 'Sequence identity (%)_first': 'Sequence identity (%)',
                                 'source_last': 'source'
                                })

af_only['frac_disorder'] = af_only['disorder_len_sum']/af_only['seq_len']

In [ ]:
s_and_a = af_only[af_only['frac_disorder'] <= 0.01].merge(unique_pdbs, on = ['ID link'], how = 'outer', indicator = True, suffixes = ['_af', '_struct'])

In [ ]:
comparisons_af= s_and_a[s_and_a._merge == 'both']

In [ ]:
#set scale to mimic seq len plot?
import plotly.express as px

df = comparisons_af.copy()

df["max_dim_struct"] = df["max_dim_struct"]/10.0

df["max_dim_af"] = df["max_dim_af"]/10.0


fig = px.scatter(df, x="max_dim_struct", y="max_dim_af", #color="source",
                 size='seq_len', 
                 color = 'mean_confidence_trimmed_first', color_continuous_scale = 'plasma_r', hover_data=[ 'ID', 'PDB'], 
                  trendline="ols",
                 width=700, height=600,
                      labels={
                     "max_dim_struct": "structural height (nm)",
                     "max_dim_af": "alphafold model height (nm)",
                     "mean_confidence_trimmed_first": "avg model confidence"
                 },
                title = 'AlphaFold Models vs. Structural Measurement'
                )
fig.update_layout(yaxis=dict(range=[0,30]), xaxis=dict(range=[0,30]))

fig.show()

results = px.get_trendline_results(fig)
print(results)

results.px_fit_results.iloc[0].summary()

#fig.write_image("alphafold_vs_structures.pdf")

https://alphafold.ebi.ac.uk/entry/P20023
deviation due to CR2 structure, alphafold often has longer proteins folded over on to themselves, due to no constraints related to the membrane bound nature of the proteins

# Domain counting only comparisons

In [ ]:
est_dom['frac_disorder'] = est_dom['disorder_len_sum']/est_dom['seq_len_first']

In [ ]:
dom_est = est_dom.sort_values('seq_len_first').groupby('ID link_first').last().reset_index()

dom_est = dom_est[(dom_est['dom_coverage'] > 0.7)&(dom_est['frac_disorder'] <= 0.01)]

dom_est.rename(columns = {'ID link_first': 'ID link', 'seq_len_first': 'seq_len', 
                                }, inplace = True)

#dom_est.dropna(axis= 1, inplace = True, how = 'all')


In [ ]:
s_and_d = dom_est.merge(unique_pdbs, on = ['ID link'], how = 'outer', indicator = True, suffixes = ['_dom', '_struct'])

In [ ]:
comparisons_dom= s_and_d[s_and_d._merge == 'both']

In [ ]:
#set scale to mimic seq len plot?
import plotly.express as px

df = comparisons_dom.copy()

df['max_dim'] = df['max_dim']/10.0

fig = px.scatter(df, x="max_dim", y="height_sum", #color="source",
                 size='seq_len', 
                 color = 'dom_len_count', color_continuous_scale = 'rainbow', hover_data=[ 'ID', 'PDB'], 
                 trendline="ols",
                 width=700, height=600,
                 labels={
                     "max_dim": "structural height (nm)",
                     "height_sum": "domain counting height (nm)",
                     "dom_len_count": "# of domains"
                 },
                title = 'Domain Counting vs. Structural Measurement')
fig.update_layout(yaxis=dict(range=[0,30]), xaxis=dict(range=[0,30]))



fig.show()

results = px.get_trendline_results(fig)
print(results)

results.px_fit_results.iloc[0].summary()

#fig.write_image("domains_vs_structures.pdf")

In [ ]:
unique_pdbs['seq_len'] = unique_pdbs['end']- unique_pdbs['start']

unique_pdbs['seq_height'] = unique_pdbs['seq_len']*0.04


df = unique_pdbs.copy()

df['max_dim'] = df['max_dim']/10.0

fig = px.scatter(df, x="max_dim", y="seq_height", #color="source",
                 size='seq_len', 
                 color = 'PDB_len', color_continuous_scale = 'rainbow', #hover_data=[ 'ID', 'PDB'], 
                 trendline="ols",
                 width=700, height=600,
                 labels={
                     "max_dim": "structural height (nm)",
                     "seq_height": "sequence len height (nm)",
                     "PDB_len": "PDB Length"
                 },
                title = 'Sequence Length vs. Structural Measurement')
fig.update_layout(yaxis=dict(range=[0,30]), xaxis=dict(range=[0,30]))



fig.show()

results = px.get_trendline_results(fig)
print(results)

results.px_fit_results.iloc[0].summary()

#fig.write_image("sequence_vs_structures.pdf")